<a href="https://colab.research.google.com/github/yasirusman85/Multimodal-Video-Insight-Search/blob/main/Multimodal_Video_Insight_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal Video Insight Search: Technical Documentation

## Project Overview
This project implements a high-performance **Multimodal Video Search Engine** that allows users to query video content using natural language. The system identifies specific moments within a video file by computing semantic similarity between a text string (the query) and temporal visual samples (extracted frames).

## 🏗️ System Architecture

The pipeline consists of three primary stages:
1. **Temporal Sampling (Frame Extraction):** Video files are decoded using `OpenCV`, sampling the stream at a configurable frequency (default 1Hz) to balance search granularity with computational efficiency.
2. **Vision-Language Embedding (CLIP):** We utilize OpenAI's `CLIP` (Contrastive Language-Image Pre-training) model (`openai/clip-vit-base-patch32`). CLIP maps both text and images into a shared high-dimensional vector space.
3. **Semantic Matching:** The cosine similarity between the query embedding and each frame embedding is calculated. The system identifies the global maximum to determine the "Best Match."

## 🛠️ Technical Specifications

### Core Technologies
- **Model:** CLIP ViT-B/32 (Vision Transformer architecture).
- **Frameworks:** `PyTorch` (Backend), `Transformers` (Model management), `Gradio` (UI/UX).
- **Processing:** `OpenCV` for frame-buffer manipulation and `PIL` for image preprocessing.

### The Moondream2 Pivot
Initially, the project explored **Moondream2**, a tiny Vision Language Model (VLM). However, due to versioning conflicts in the `transformers` library environment regarding `rope_scaling` and `pad_token_id` within custom modeling files, the architecture was pivoted to **CLIP**.

**Advantages of the CLIP switch:**
- **Stability:** Native support in the `transformers` library without requiring manual monkey-patching of remote code.
- **Performance:** Optimized for zero-shot retrieval tasks, providing faster inference than full-causal autoregressive VLMs.
- **Scalability:** Lower memory footprint (running on `float16`), allowing for longer video processing on standard GPU runtimes.

## 🧪 Inference Pipeline Detail

```python
# Mathematical logic for similarity scoring
# Given Query Q and Frames F[n]:
# Score[i] = CosineSimilarity(Embedding(Q), Embedding(F[i]))
# Result = Max(Score)
```

1. **Preprocessing:** Frames are resized to 224x224 and normalized per ImageNet statistics.
2. **Encoding:** Both modalities are passed through their respective encoders to generate 512-dimension vectors.
3. **Post-processing:** Softmax is applied across the temporal dimension of the similarity logits to derive a confidence percentage.

## 🚀 Deployment
The interface is deployed via **Gradio**, providing a localized tunnel for video uploads and real-time visualization of the search results with associated timestamps.

# 🎥 Multimodal Video Insight Search
This notebook demonstrates using the **Moondream2** Vision Language Model (VLM) to search for specific moments within a video using natural language queries.

In [25]:
!pip install -q gradio transformers torch opencv-python pillow

## 🛠️ Step 1: Utility Functions
We define a function to extract frames from the uploaded video at set intervals.

In [2]:
import cv2
from PIL import Image

def extract_frames(video_path, interval=1):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = []
    timestamps = []

    count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if count % int(fps * interval) == 0:
            # Convert BGR to RGB
            color_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(color_frame))
            timestamps.append(count / fps)
        count += 1
    cap.release()
    return frames, timestamps

## 🧠 Step 2: Model Loading
We use **OpenAI's CLIP** model for zero-shot image-text matching. This allows us to compare the user's text query directly against video frames to find the best match.

In [26]:
from transformers import CLIPProcessor, CLIPModel
import torch

# Load the CLIP model - standard and very stable for zero-shot image search
model_id = "openai/clip-vit-base-patch32"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)

print(f"✅ CLIP Model loaded successfully on {device}!")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

✅ CLIP Model loaded successfully on cuda!


In [27]:
def analyze_video(video_path, user_query):
    frames, timestamps = extract_frames(video_path, interval=1)
    if not frames:
        return None, "Could not extract frames from video."

    # Process the text query
    inputs = processor(text=[user_query], images=frames, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        # Get similarity scores between the text and each frame
        logits_per_image = outputs.logits_per_image
        probs = logits_per_image.softmax(dim=0).cpu().numpy()

    # Find the index of the frame with the highest similarity
    best_match_idx = probs.argmax()
    best_score = probs[best_match_idx][0]

    if best_score < 0.01: # Threshold to ensure some relevance
        return None, "No strong match found for your query."

    best_frame = frames[best_match_idx]
    timestamp = timestamps[best_match_idx]

    return best_frame, f"Best match found at {timestamp:.2f}s (Confidence: {best_score:.2%})"

## 🚀 Step 3: Launch Interface
Upload a video and type what you are looking for to find the specific moment.

In [33]:
import gradio as gr

# Ensure we launch the demo with the updated analyze_video function
with gr.Blocks() as demo:
    gr.Markdown("# 🎥 Multimodal Video Insight Search")
    gr.Markdown("Search for specific moments in your video using natural language descriptions.")

    with gr.Row():
        video_input = gr.Video(label="1. Upload Video")
        query_input = gr.Textbox(label="2. What are you looking for?", placeholder="e.g., a person in a red hat")

    search_btn = gr.Button("🔍 Search Video", variant="primary")

    with gr.Row():
        image_output = gr.Image(label="Best Match Found", type="pil")
        text_output = gr.Textbox(label="Result Details")

    search_btn.click(analyze_video, inputs=[video_input, query_input], outputs=[image_output, text_output])

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().


KeyboardInterrupt: 

## 🛠️ GitHub Compatibility Fix
If you are seeing the error `the 'state' key is missing from 'metadata.widgets'` when uploading to GitHub, run the cell below. It will generate a 'cleaned' version of this notebook in the file explorer that you can download and upload to GitHub.

In [34]:
import json
import nbformat
from google.colab import files

def clean_notebook_for_github(input_filename, output_filename):
    with open(input_filename, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    # Remove the problematic widgets metadata
    if 'metadata' in nb and 'widgets' in nb['metadata']:
        print("Found problematic 'widgets' metadata. Removing...")
        del nb['metadata']['widgets']

    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(nb, f, indent=1)

    print(f"✅ Cleaned notebook saved as: {output_filename}")

# Note: In Colab, the current notebook filename isn't easily accessible via code
# You can manually upload the .ipynb file to the sidebar, or use the snippet below
print("Please download this notebook as .ipynb first, then upload it to the Colab file sidebar.")
# clean_notebook_for_github('your_filename.ipynb', 'github_ready.ipynb')


Please download this notebook as .ipynb first, then upload it to the Colab file sidebar.
